In [56]:
import importlib
import configl
importlib.reload(configl)

import feature_extraction
importlib.reload(feature_extraction)

<module 'feature_extraction' from '/home/mariana/proyecto_PAT/feature_extraction.py'>

In [57]:
# ═══════════════════════════════════════════════════════════
# CELDA 1: Imports y recarga de módulos
# ═══════════════════════════════════════════════════════════
import glob
import json
import os
import pickle
import numpy as np
import importlib

import configl
importlib.reload(configl)

from feature_extraction import EmotionExtractor

print("Fandoms:", configl.FANDOMS)
print("Tropes:", configl.TROPES)
print(f"Modelo: {configl.ROBERTA_MODEL}")
print(f"Embeddings: {configl.ROBERTA_EMBED_DIM}d por ventana")
print(f"Solapamiento: {configl.CHUNK_OVERLAP_RATIO*100:.0f}% ({configl.CHUNK_OVERLAP_TOKENS} tokens)")

Fandoms: ['One Direction', 'BTS', 'Harry Potter']
Tropes: ['hurt_comfort', 'fluff', 'slow_burn']
Modelo: SamLowe/roberta-base-go_emotions
Embeddings: 768d por ventana
Solapamiento: 10% (51 tokens)


In [58]:
# ═══════════════════════════════════════════════════════════
# CELDA 2: Cargar JSONs y filtrar fandoms/tropes válidos
# ═══════════════════════════════════════════════════════════
raw_dir = "/home/mariana/proyecto_PAT/data/raw/"
archivos = glob.glob(os.path.join(raw_dir, "**/*.json"), recursive=True)
print(f"Archivos JSON encontrados: {len(archivos)}")

# Fandoms y tropes válidos según config actualizado
valid_fandoms = set(config.FANDOMS.keys()) if isinstance(config.FANDOMS, dict) else set(config.FANDOMS)
valid_tropes = set(config.TROPES.keys()) if isinstance(config.TROPES, dict) else set(config.TROPES)

dataset = []
skipped = {"fandom": 0, "trope": 0, "no_comments": 0}

for filepath in archivos:
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    fandom = data["fandom_key"]
    trope = data["trope_key"]

    # Filtrar fandoms/tropes válidos (sin twilight ni whump)
    if fandom not in valid_fandoms:
        skipped["fandom"] += 1
        continue
    if trope not in valid_tropes:
        skipped["trope"] += 1
        continue

    # Extraer texto de capítulos y comentarios
    chapters_text = []
    comments_by_chapter = []

    for ch in data["chapters"]:
        chapters_text.append(ch["text"])

        # Filtrar comentarios: mínimo de palabras para riqueza emocional
        good_comments = [
            c["text"] for c in ch["comments"]
            if len(c["text"].split()) >= config.MIN_COMMENT_WORDS
        ]
        comments_by_chapter.append(good_comments)

    # Verificar que haya comentarios suficientes
    total_comments = sum(len(c) for c in comments_by_chapter)
    if total_comments == 0:
        skipped["no_comments"] += 1
        continue

    dataset.append({
        "work_id": data["work_id"],
        "title": data["title"],
        "fandom": fandom,
        "trope": trope,
        "kudos": data["kudos"],
        "chapters": chapters_text,
        "comments_by_chapter": comments_by_chapter,
        "num_chapters": len(chapters_text)
    })

print(f"\n✓ Fanfics cargados: {len(dataset)}")
print(f"Descartados: {skipped}")

# Distribución por celda
from collections import Counter
conteo = Counter((d["fandom"], d["trope"]) for d in dataset)
print(f"\nDistribución:")
for (fandom, trope), count in sorted(conteo.items()):
    print(f"  {fandom:20s} / {trope:15s} : {count}")

Archivos JSON encontrados: 385

✓ Fanfics cargados: 270
Descartados: {'fandom': 61, 'trope': 54, 'no_comments': 0}

Distribución:
  bts                  / fluff           : 30
  bts                  / hurt_comfort    : 30
  bts                  / slow_burn       : 30
  harry_potter         / fluff           : 30
  harry_potter         / hurt_comfort    : 30
  harry_potter         / slow_burn       : 30
  one_direction        / fluff           : 30
  one_direction        / hurt_comfort    : 30
  one_direction        / slow_burn       : 30


In [59]:
# ═══════════════════════════════════════════════════════════
# CELDA 3: Inspeccionar una muestra antes de procesar
# ═══════════════════════════════════════════════════════════
sample = dataset[0]
print(f"Título: {sample['title']}")
print(f"Fandom: {sample['fandom']} | Trope: {sample['trope']}")
print(f"Kudos: {sample['kudos']}")
print(f"Capítulos: {sample['num_chapters']}")
print(f"\nPor capítulo:")
for i, (ch, comm) in enumerate(zip(sample["chapters"], sample["comments_by_chapter"])):
    n_words = len(ch.split())
    n_tokens_approx = int(n_words * 1.3)  # Aproximación tokens
    n_windows = max(1, (n_tokens_approx - configl.CHUNK_OVERLAP_TOKENS) // (configl.ROBERTA_MAX_TOKENS - configl.CHUNK_OVERLAP_TOKENS) + 1)
    print(f"  Cap {i+1}: {n_words} palabras ≈ {n_windows} ventanas | {len(comm)} comentarios válidos")

Título: Sex and the Art of Castle Maintenance
Fandom: harry_potter | Trope: hurt_comfort
Kudos: 11979
Capítulos: 1

Por capítulo:
  Cap 1: 15059 palabras ≈ 43 ventanas | 20 comentarios válidos


In [60]:
# ═══════════════════════════════════════════════════════════
# CELDA 4: Extraer features emocionales
# ⚠ TARDA ~2-4 HORAS dependiendo de tu GPU/CPU
# Guarda checkpoint cada 25 fanfics por si se interrumpe
# ═══════════════════════════════════════════════════════════
output_dir = "/home/mariana/proyecto_PAT/data/features/"
os.makedirs(output_dir, exist_ok=True)
checkpoint_path = os.path.join(output_dir, "checkpoint_features.pkl")
final_path = os.path.join(output_dir, "emotional_features.pkl")

# Si ya hay un checkpoint, reanudar desde ahí
start_idx = 0
features = []
if os.path.exists(checkpoint_path):
    with open(checkpoint_path, "rb") as f:
        features = pickle.load(f)
    start_idx = len(features)
    print(f"Reanudando desde checkpoint: {start_idx}/{len(dataset)} procesados")

# Inicializar extractor
extractor = EmotionExtractor()

for idx in range(start_idx, len(dataset)):
    fanfic = dataset[idx]
    print(f"\n[{idx+1}/{len(dataset)}] {fanfic['title'][:60]}")
    print(f"  {fanfic['fandom']}/{fanfic['trope']} | {fanfic['num_chapters']} caps")

    # ── Capítulos: embeddings [CLS] por ventana → [n_ventanas, 768] ──
    print("  → Extrayendo embeddings de capítulos...")
    chapter_embeddings = extractor.extract_chapter_arc(fanfic["chapters"])

    # ── Comentarios: emoción dominante → distribución por capítulo [28] ──
    print("  → Clasificando comentarios...")
    reader_arc = extractor.extract_comment_arc(fanfic["comments_by_chapter"])

    features.append({
        "chapter_embeddings": chapter_embeddings,  # lista de arrays [n_ventanas_i, 768]
        "reader_arc": reader_arc,                   # array [n_caps, 28]
        "trope": configl.TROPE_TO_IDX[fanfic["trope"]],
        "fandom": fanfic["fandom"],
        "title": fanfic["title"],
        "num_chapters": fanfic["num_chapters"]
    })

    # Checkpoint cada 25 fanfics
    if (idx + 1) % 25 == 0:
        with open(checkpoint_path, "wb") as f:
            pickle.dump(features, f)
        print(f"  💾 Checkpoint guardado ({idx+1}/{len(dataset)})")

# Guardar features finales
with open(final_path, "wb") as f:
    pickle.dump(features, f)
print(f"\n{'='*50}")
print(f"✓ Features completos: {len(features)} fanfics")
print(f"  Guardados en: {final_path}")

Cargando SamLowe/roberta-base-go_emotions en cpu...


Loading weights: 100%|█████████████████████| 201/201 [00:00<00:00, 13869.23it/s]
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (20149 > 512). Running this sequence through the model will result in indexing errors



[1/270] Sex and the Art of Castle Maintenance
  harry_potter/hurt_comfort | 1 caps
  → Extrayendo embeddings de capítulos...
    Cap 1/1: 15059 palabras → 44 ventanas
  → Clasificando comentarios...
    Comentarios cap 1: 20 → emoción dominante: admiration

[2/270] Like Real People Do
  harry_potter/hurt_comfort | 3 caps
  → Extrayendo embeddings de capítulos...
    Cap 1/3: 7087 palabras → 23 ventanas
    Cap 2/3: 12454 palabras → 42 ventanas
    Cap 3/3: 17062 palabras → 55 ventanas
  → Clasificando comentarios...
    Comentarios cap 1: 20 → emoción dominante: love
    Comentarios cap 2: 20 → emoción dominante: love
    Comentarios cap 3: 20 → emoción dominante: love

[3/270] Lead Me Into the Light
  harry_potter/hurt_comfort | 2 caps
  → Extrayendo embeddings de capítulos...
    Cap 1/2: 11150 palabras → 38 ventanas
    Cap 2/2: 6586 palabras → 22 ventanas
  → Clasificando comentarios...
    Comentarios cap 1: 20 → emoción dominante: love
    Comentarios cap 2: 20 → emoción dominan

In [5]:
import pickle
import numpy as np

with open("data/features/emotional_features.pkl", "rb") as f:
    features = pickle.load(f)

print(f"Total fanfics: {len(features)}")
print(f"\n{'='*60}")
print("KEYS disponibles en cada fanfic:")
print(f"  {list(features[0].keys())}")

# ── Verificar Uso 1: embeddings de capítulos ──
print(f"\n{'='*60}")
print("USO 1 — Embeddings de capítulos [n_ventanas, 768]")
r = features[0]
has_chapter_emb = "chapter_embeddings" in r
print(f"  ¿Tiene chapter_embeddings? {has_chapter_emb}")
if has_chapter_emb:
    print(f"  Tipo: {type(r['chapter_embeddings'])}")
    print(f"  Capítulos: {len(r['chapter_embeddings'])}")
    print(f"  Ventanas cap 1: {r['chapter_embeddings'][0].shape}")
    print(f"  ✅ Uso 1 OK" if r['chapter_embeddings'][0].shape[1] == 768 else "  ❌ Dimensión incorrecta")

# ── Verificar Uso 2: embeddings de comentarios ──
print(f"\n{'='*60}")
print("USO 2 — Embeddings de comentarios [1, 768] por comentario")
has_comment_emb = "comment_embeddings" in r
print(f"  ¿Tiene comment_embeddings? {has_comment_emb}")
if has_comment_emb:
    print(f"  Tipo: {type(r['comment_embeddings'])}")
    print(f"  Capítulos con embeddings: {len(r['comment_embeddings'])}")
else:
    print(f"  ❌ FALTA — necesitas extraer embeddings de comentarios")
    print(f"     (el LSTM comment encoder necesita [1, 768] por comentario)")

# ── Verificar Uso 3: clasificación de comentarios (reader_arc) ──
print(f"\n{'='*60}")
print("USO 3 — Clasificación de comentarios [n_caps, 28]")
has_reader_arc = "reader_arc" in r
print(f"  ¿Tiene reader_arc? {has_reader_arc}")
if has_reader_arc:
    print(f"  Shape: {r['reader_arc'].shape}")
    print(f"  Rango de valores: [{r['reader_arc'].min():.4f}, {r['reader_arc'].max():.4f}]")
    print(f"  ✅ Uso 3 OK" if r['reader_arc'].shape[1] == 28 else "  ❌ Dimensión incorrecta")

# ── Verificar metadata ──
print(f"\n{'='*60}")
print("METADATA")
print(f"  Trope: {r['trope']} (tipo: {type(r['trope'])})")
print(f"  Fandom: {r['fandom']}")
print(f"  Título: {r['title']}")
print(f"  Capítulos: {r['num_chapters']}")

Total fanfics: 270

KEYS disponibles en cada fanfic:
  ['chapter_embeddings', 'reader_arc', 'trope', 'fandom', 'title', 'num_chapters', 'comment_embeddings']

USO 1 — Embeddings de capítulos [n_ventanas, 768]
  ¿Tiene chapter_embeddings? True
  Tipo: <class 'list'>
  Capítulos: 1
  Ventanas cap 1: (44, 768)
  ✅ Uso 1 OK

USO 2 — Embeddings de comentarios [1, 768] por comentario
  ¿Tiene comment_embeddings? True
  Tipo: <class 'list'>
  Capítulos con embeddings: 1

USO 3 — Clasificación de comentarios [n_caps, 28]
  ¿Tiene reader_arc? True
  Shape: (1, 28)
  Rango de valores: [0.0000, 0.4500]
  ✅ Uso 3 OK

METADATA
  Trope: 0 (tipo: <class 'int'>)
  Fandom: harry_potter
  Título: Sex and the Art of Castle Maintenance
  Capítulos: 1


In [4]:
# ═══════════════════════════════════════════════════════════
# Versión RÁPIDA: procesa comentarios en batches
# ═══════════════════════════════════════════════════════════
import pickle, json, glob, os
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "SamLowe/roberta-base-go_emotions"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, output_hidden_states=True
).to(device).eval()

for p in model.parameters():
    p.requires_grad = False

with open("data/features/emotional_features.pkl", "rb") as f:
    features = pickle.load(f)

# Indexar JSONs
raw_dir = "data/raw/"
json_index = {}
for fp in glob.glob(os.path.join(raw_dir, "**/*.json"), recursive=True):
    with open(fp, "r", encoding="utf-8") as f:
        d = json.load(f)
    json_index[d["title"]] = d

@torch.no_grad()
def batch_embed_comments(texts, batch_size=16):
    """Procesa múltiples comentarios de una vez → mucho más rápido."""
    if not texts:
        return np.zeros((1, 768), dtype=np.float32)
    
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(
            batch, return_tensors="pt", truncation=True,
            max_length=512, padding=True
        ).to(device)
        outputs = model(**inputs)
        # [CLS] de cada comentario en el batch
        cls = outputs.hidden_states[-1][:, 0, :].cpu().numpy()
        all_embs.append(cls)
    
    all_embs = np.concatenate(all_embs, axis=0)  # (n_comments, 768)
    # Promediar todos → 1 vector por capítulo
    return all_embs.mean(axis=0, keepdims=True).astype(np.float32)  # (1, 768)

missing = 0
for idx, feat in enumerate(features):
    title = feat["title"]
    
    if title not in json_index:
        feat["comment_embeddings"] = [
            np.zeros((1, 768), dtype=np.float32)
            for _ in range(feat["num_chapters"])
        ]
        missing += 1
        continue
    
    raw = json_index[title]
    comment_embs = []
    
    for ch in raw["chapters"]:
        good = [c["text"] for c in ch["comments"] if len(c["text"].split()) >= 15]
        avg_emb = batch_embed_comments(good, batch_size=16)
        comment_embs.append(avg_emb)
    
    feat["comment_embeddings"] = comment_embs
    
    if (idx + 1) % 50 == 0:
        print(f"[{idx+1}/{len(features)}] ✓")
        with open("data/features/emotional_features.pkl", "wb") as f:
            pickle.dump(features, f)

with open("data/features/emotional_features.pkl", "wb") as f:
    pickle.dump(features, f)

print(f"\n✅ Listo: {len(features)-missing}/{len(features)} fanfics con comment_embeddings")
print(f"Verificación: {features[0]['comment_embeddings'][0].shape}")  # (1, 768)

Loading weights: 100%|███████████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 16045.05it/s]


[50/270] ✓
[100/270] ✓
[150/270] ✓
[200/270] ✓
[250/270] ✓

✅ Listo: 270/270 fanfics con comment_embeddings
Verificación: (1, 768)


In [6]:
problemas = []
stats = {"caps": [], "ventanas": [], "emb_dim": set()}

for i, f in enumerate(features):
    n_caps = f["num_chapters"]
    
    # Uso 1: chapter_embeddings
    if "chapter_embeddings" in f:
        if len(f["chapter_embeddings"]) != n_caps:
            problemas.append(f"Fanfic {i}: chapter_embeddings tiene {len(f['chapter_embeddings'])} pero num_chapters={n_caps}")
        for c, emb in enumerate(f["chapter_embeddings"]):
            stats["ventanas"].append(emb.shape[0])
            stats["emb_dim"].add(emb.shape[1])
    
    # Uso 3: reader_arc
    if "reader_arc" in f:
        if f["reader_arc"].shape[0] != n_caps:
            problemas.append(f"Fanfic {i}: reader_arc tiene {f['reader_arc'].shape[0]} filas pero num_chapters={n_caps}")
    
    # Uso 2: comment_embeddings
    if "comment_embeddings" not in f:
        if i == 0:
            problemas.append("GLOBAL: comment_embeddings NO existe en los features")
    
    stats["caps"].append(n_caps)

print(f"Fanfics verificados: {len(features)}")
print(f"Capítulos: min={min(stats['caps'])}, max={max(stats['caps'])}, media={np.mean(stats['caps']):.1f}")
print(f"Ventanas por capítulo: min={min(stats['ventanas'])}, max={max(stats['ventanas'])}, media={np.mean(stats['ventanas']):.1f}")
print(f"Dimensiones de embedding: {stats['emb_dim']}")
print(f"\nProblemas encontrados: {len(problemas)}")
for p in problemas:
    print(f"  ⚠ {p}")

Fanfics verificados: 270
Capítulos: min=1, max=25, media=8.9
Ventanas por capítulo: min=1, max=189, media=28.0
Dimensiones de embedding: {768}

Problemas encontrados: 0


In [7]:
EMOTION_LABELS = [
    "admiration", "amusement", "anger", "annoyance", "approval",
    "caring", "confusion", "curiosity", "desire", "disappointment",
    "disapproval", "disgust", "embarrassment", "excitement", "fear",
    "gratitude", "grief", "joy", "love", "nervousness",
    "optimism", "pride", "realization", "relief", "remorse",
    "sadness", "surprise", "neutral"
]

EMOTION_TO_CATEGORY = {
    "admiration": 0, "amusement": 0, "approval": 0, "caring": 0,
    "desire": 0, "excitement": 0, "gratitude": 0, "joy": 0,
    "love": 0, "optimism": 0, "pride": 0, "relief": 0,
    "anger": 1, "annoyance": 1, "disappointment": 1, "disapproval": 1,
    "disgust": 1, "embarrassment": 1, "fear": 1, "grief": 1,
    "nervousness": 1, "remorse": 1, "sadness": 1,
    "confusion": 2, "curiosity": 2, "realization": 2, "surprise": 2,
    "neutral": 2
}

CAT_NAMES = {0: "positivo", 1: "negativo", 2: "ambiguo"}

def collapse_to_categories(reader_arc):
    n_caps = reader_arc.shape[0]
    cat_probs = np.zeros((n_caps, 3))
    for emo_idx, emo_name in enumerate(EMOTION_LABELS):
        cat = EMOTION_TO_CATEGORY[emo_name]
        cat_probs[:, cat] += reader_arc[:, emo_idx]
    return cat_probs.argmax(axis=1)

# Distribución global
all_cats = []
cats_by_trope = {0: [], 1: [], 2: []}

for f in features:
    cats = collapse_to_categories(f["reader_arc"])
    all_cats.extend(cats)
    for c in cats:
        cats_by_trope[f["trope"]].append(c)

from collections import Counter

print("Distribución GLOBAL de categorías emocionales en comentarios:")
dist = Counter(all_cats)
total = len(all_cats)
for cat_idx in [0, 1, 2]:
    pct = dist[cat_idx] / total * 100
    bar = "█" * int(pct / 2)
    print(f"  {CAT_NAMES[cat_idx]:10s}: {dist[cat_idx]:4d} ({pct:5.1f}%) {bar}")

print(f"\n  Total capítulos-comentario: {total}")

if dist.most_common(1)[0][1] / total > 0.80:
    print(f"\n  ⚠ DESBALANCE SEVERO — la clase '{CAT_NAMES[dist.most_common(1)[0][0]]}' domina >80%")
    print(f"    → Necesitas cross-entropy PONDERADA en Tarea 2")
else:
    print(f"\n  ✅ Balance aceptable para cross-entropy estándar")

# Por trope
TROPE_NAMES = {0: "hurt_comfort", 1: "fluff", 2: "slow_burn"}
print(f"\nDistribución POR TROPE:")
for t_idx in [0, 1, 2]:
    t_dist = Counter(cats_by_trope[t_idx])
    t_total = len(cats_by_trope[t_idx])
    print(f"\n  {TROPE_NAMES[t_idx]}:")
    for cat_idx in [0, 1, 2]:
        pct = t_dist[cat_idx] / t_total * 100 if t_total > 0 else 0
        print(f"    {CAT_NAMES[cat_idx]:10s}: {t_dist[cat_idx]:3d} ({pct:5.1f}%)")

Distribución GLOBAL de categorías emocionales en comentarios:
  positivo  : 2367 ( 98.1%) █████████████████████████████████████████████████
  negativo  :   16 (  0.7%) 
  ambiguo   :   30 (  1.2%) 

  Total capítulos-comentario: 2413

  ⚠ DESBALANCE SEVERO — la clase 'positivo' domina >80%
    → Necesitas cross-entropy PONDERADA en Tarea 2

Distribución POR TROPE:

  hurt_comfort:
    positivo  : 803 ( 98.2%)
    negativo  :   8 (  1.0%)
    ambiguo   :   7 (  0.9%)

  fluff:
    positivo  : 473 ( 99.2%)
    negativo  :   2 (  0.4%)
    ambiguo   :   2 (  0.4%)

  slow_burn:
    positivo  : 1091 ( 97.6%)
    negativo  :   6 (  0.5%)
    ambiguo   :  21 (  1.9%)


In [8]:
import numpy as np

caps = [f["num_chapters"] for f in features]
wins = [emb.shape[0] for f in features for emb in f["chapter_embeddings"]]

print("CAPÍTULOS por fanfic:")
for threshold in [10, 15, 20, 25]:
    count = sum(1 for c in caps if c > threshold)
    print(f"  >{threshold}: {count} fanfics ({count/len(caps)*100:.1f}%)")

print(f"\nVENTANAS por capítulo:")
for threshold in [30, 50, 80, 100, 150]:
    count = sum(1 for w in wins if w > threshold)
    print(f"  >{threshold}: {count} capítulos ({count/len(wins)*100:.1f}%)")

# Verificar comment_embeddings
r = features[0]
has_ce = "comment_embeddings" in r
print(f"\n✅ comment_embeddings presente: {has_ce}")
if has_ce:
    print(f"  Shape: {r['comment_embeddings'][0].shape}")

CAPÍTULOS por fanfic:
  >10: 108 fanfics (40.0%)
  >15: 53 fanfics (19.6%)
  >20: 23 fanfics (8.5%)
  >25: 0 fanfics (0.0%)

VENTANAS por capítulo:
  >30: 799 capítulos (33.1%)
  >50: 252 capítulos (10.4%)
  >80: 69 capítulos (2.9%)
  >100: 36 capítulos (1.5%)
  >150: 7 capítulos (0.3%)

✅ comment_embeddings presente: True
  Shape: (1, 768)


In [9]:
import numpy as np

EMOTION_LABELS = [
    "admiration", "amusement", "anger", "annoyance", "approval",
    "caring", "confusion", "curiosity", "desire", "disappointment",
    "disapproval", "disgust", "embarrassment", "excitement", "fear",
    "gratitude", "grief", "joy", "love", "nervousness",
    "optimism", "pride", "realization", "relief", "remorse",
    "sadness", "surprise", "neutral"
]

TROPE_NAMES = {0: "hurt_comfort", 1: "fluff", 2: "slow_burn"}

# Promedio de las 28 emociones por trope
for t_idx in [0, 1, 2]:
    arcs = [f["reader_arc"] for f in features if f["trope"] == t_idx]
    # Concatenar todos los capítulos de ese trope
    all_caps = np.concatenate(arcs, axis=0)  # (total_caps, 28)
    mean_dist = all_caps.mean(axis=0)
    
    # Top 5 emociones para este trope
    top5 = mean_dist.argsort()[-5:][::-1]
    
    print(f"\n{TROPE_NAMES[t_idx]} — top 5 emociones en comentarios:")
    for rank, idx in enumerate(top5):
        bar = "█" * int(mean_dist[idx] * 100)
        print(f"  {rank+1}. {EMOTION_LABELS[idx]:15s}: {mean_dist[idx]:.3f} {bar}")


hurt_comfort — top 5 emociones en comentarios:
  1. love           : 0.280 ███████████████████████████
  2. admiration     : 0.199 ███████████████████
  3. gratitude      : 0.186 ██████████████████
  4. sadness        : 0.047 ████
  5. amusement      : 0.042 ████

fluff — top 5 emociones en comentarios:
  1. love           : 0.303 ██████████████████████████████
  2. admiration     : 0.213 █████████████████████
  3. gratitude      : 0.166 ████████████████
  4. amusement      : 0.062 ██████
  5. neutral        : 0.036 ███

slow_burn — top 5 emociones en comentarios:
  1. love           : 0.269 ██████████████████████████
  2. admiration     : 0.181 ██████████████████
  3. gratitude      : 0.160 ████████████████
  4. neutral        : 0.062 ██████
  5. amusement      : 0.059 █████


In [1]:
import pickle, json, glob, os, torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = "cpu"
model_name = "SamLowe/roberta-base-go_emotions"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device).eval()
for p in model.parameters():
    p.requires_grad = False

with open("data/features/emotional_features.pkl", "rb") as f:
    features = pickle.load(f)

# Indexar JSONs
raw_dir = "data/raw/"
json_index = {}
for fp in glob.glob(os.path.join(raw_dir, "**/*.json"), recursive=True):
    with open(fp, "r", encoding="utf-8") as f:
        d = json.load(f)
    json_index[d["title"]] = d

@torch.no_grad()
def classify_batch(texts, batch_size=16):
    """Clasifica textos → distribución de 28 emociones (sigmoid, no argmax)."""
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", truncation=True,
                          max_length=512, padding=True).to(device)
        probs = torch.sigmoid(model(**inputs).logits).cpu().numpy()
        all_probs.append(probs)
    return np.concatenate(all_probs, axis=0)  # (n_texts, 28)

def chunk_text(text, max_tokens=510, overlap=51):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    step = max_tokens - overlap
    chunks = []
    for start in range(0, len(tokens), step):
        chunk = tokens[start:start + max_tokens]
        chunks.append(tokenizer.decode(chunk, skip_special_tokens=True))
        if start + max_tokens >= len(tokens):
            break
    return chunks if chunks else [text]

# Procesar cada fanfic
for idx, feat in enumerate(features):
    title = feat["title"]
    if title not in json_index:
        feat["writer_arc"] = np.zeros((feat["num_chapters"], 28), dtype=np.float32)
        continue
    
    raw = json_index[title]
    chapter_emotions = []
    
    for ch in raw["chapters"]:
        chunks = chunk_text(ch["text"])
        # Distribución COMPLETA por ventana (28 probabilidades, no argmax)
        chunk_probs = classify_batch(chunks)       # (n_chunks, 28)
        # Promediar ventanas → distribución del capítulo
        chapter_dist = chunk_probs.mean(axis=0)     # (28,)
        chapter_emotions.append(chapter_dist)
    
    feat["writer_arc"] = np.array(chapter_emotions, dtype=np.float32)  # (n_caps, 28)
    
    if (idx + 1) % 25 == 0:
        print(f"[{idx+1}/{len(features)}] ✓")
        with open("data/features/emotional_features.pkl", "wb") as f:
            pickle.dump(features, f)

with open("data/features/emotional_features.pkl", "wb") as f:
    pickle.dump(features, f)

print(f"\n✅ writer_arc extraído: {features[0]['writer_arc'].shape}")  # (n_caps, 28)

/home/mariana/miniforge3/envs/marbio/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████| 201/201 [00:00<00:00, 13579.27it/s]
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (20149 > 512). Running this sequence through the model will result in indexing errors


[25/270] ✓
[50/270] ✓
[75/270] ✓
[100/270] ✓
[125/270] ✓
[150/270] ✓
[175/270] ✓
[200/270] ✓
[225/270] ✓
[250/270] ✓

✅ writer_arc extraído: (1, 28)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import configl

EMOTION_LABELS = configl.EMOTION_LABELS
TROPE_NAMES = {0: "hurt_comfort", 1: "fluff", 2: "slow_burn"}

fig, axes = plt.subplots(3, 2, figsize=(18, 16))
target_len = 15

for t_idx in range(3):
    writer_arcs = [f["writer_arc"] for f in features if f["trope"] == t_idx]
    reader_arcs = [f["reader_arc"] for f in features if f["trope"] == t_idx]
    
    # Interpolar a longitud común
    def interpolate_arcs(arcs):
        interp = []
        for arc in arcs:
            if len(arc) < 2:
                continue
            x_old = np.linspace(0, 1, len(arc))
            x_new = np.linspace(0, 1, target_len)
            i = np.array([np.interp(x_new, x_old, arc[:, e]) for e in range(28)]).T
            interp.append(i)
        return np.mean(interp, axis=0) if interp else np.zeros((target_len, 28))
    
    mean_writer = interpolate_arcs(writer_arcs)
    mean_reader = interpolate_arcs(reader_arcs)
    
    # Top 5 emociones con mayor varianza temporal en el ESCRITOR
    var_writer = mean_writer.var(axis=0)
    top5 = var_writer.argsort()[-5:][::-1]
    
    colores = ["#E24B4A", "#534AB7", "#0F6E56", "#D85A30", "#666666"]
    x = np.arange(1, target_len + 1)
    t_name = TROPE_NAMES[t_idx].replace("_", " ").title()
    
    # Panel izquierdo: ESCRITOR
    ax = axes[t_idx][0]
    for emo_idx, color in zip(top5, colores):
        ax.plot(x, mean_writer[:, emo_idx], "-o", color=color,
               label=EMOTION_LABELS[emo_idx], linewidth=2, markersize=4)
    ax.set_title(f"{t_name} — ESCRITOR (texto)", fontsize=13, fontweight="bold")
    ax.set_ylabel("Intensidad emocional")
    ax.set_xlabel("Progresión narrativa")
    ax.legend(fontsize=8)
    ax.set_ylim(0, min(1.0, mean_writer[:, top5].max() * 1.3))
    ax.grid(True, alpha=0.3)
    
    # Panel derecho: LECTOR
    ax = axes[t_idx][1]
    for emo_idx, color in zip(top5, colores):
        ax.plot(x, mean_reader[:, emo_idx], "s--", color=color,
               label=EMOTION_LABELS[emo_idx], linewidth=2, markersize=4)
    ax.set_title(f"{t_name} — LECTOR (comentarios)", fontsize=13, fontweight="bold")
    ax.set_ylabel("Intensidad emocional")
    ax.set_xlabel("Progresión narrativa")
    ax.legend(fontsize=8)
    ax.set_ylim(0, min(1.0, mean_reader[:, top5].max() * 1.3))
    ax.grid(True, alpha=0.3)

fig.suptitle("Arcos emocionales: ESCRITOR vs LECTOR por trope\n"
             "(distribución completa de 28 emociones, top 5 con mayor variación)",
             fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("results/writer_vs_reader_arcs.png", dpi=150, bbox_inches="tight")
plt.show()